# Lab | BabyAGI with agent

**Change the planner objective below by changing the objective and the associated prompts and potential tools and agents - Wear your creativity and AI engineering hats
You can't get this wrong!**

You would need the OpenAI API KEY and the [SerpAPI KEY](https://serpapi.com/manage-api-keyhttps://serpapi.com/manage-api-key) to run this lab.


## BabyAGI with Tools

This notebook builds on top of [baby agi](baby_agi.html), but shows how you can swap out the execution chain. The previous execution chain was just an LLM which made stuff up. By swapping it out with an agent that has access to tools, we can hopefully get real reliable information

## Install and Import Required Modules

In [ ]:
!pip install -q langchain_community langchain_experimental langchain_openai faiss-cpu google-search-results python-dotenv


In [5]:
from typing import Optional

from langchain.chains import LLMChain
from langchain.prompts import PromptTemplate
from langchain_experimental.autonomous_agents import BabyAGI
from langchain_openai import OpenAI, OpenAIEmbeddings

## Connect to the Vector Store

Depending on what vectorstore you use, this step may look different.

In [7]:
# %pip install faiss-cpu > /dev/null
# %pip install google-search-results > /dev/null
from langchain.docstore import InMemoryDocstore
from langchain_community.vectorstores import FAISS

In [8]:
import os
from dotenv import load_dotenv, find_dotenv

_ = load_dotenv(find_dotenv())

OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")
SERPAPI_API_KEY = os.getenv("SERPAPI_API_KEY")

if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY is missing. Add it to your .env file or notebook environment.")

if not SERPAPI_API_KEY:
    raise ValueError("SERPAPI_API_KEY is missing. Add it to your .env file or notebook environment.")


In [9]:
# Define the embedding model used by BabyAGI's task-result memory.
embeddings_model = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=OPENAI_API_KEY,
)

# text-embedding-3-small produces 1536-dimensional vectors.
import faiss

embedding_size = 1536
index = faiss.IndexFlatL2(embedding_size)

vectorstore = FAISS(
    embeddings_model.embed_query,
    index,
    InMemoryDocstore({}),
    {},
)


`embedding_function` is expected to be an Embeddings object, support for passing in a function will soon be removed.


## Define the Chains

BabyAGI relies on three LLM chains:
- Task creation chain to select new tasks to add to the list
- Task prioritization chain to re-prioritize tasks
- Execution Chain to execute the tasks


NOTE: in this notebook, the Execution chain will now be an agent.

In [11]:
from langchain.agents import AgentExecutor, Tool, ZeroShotAgent
from langchain.chains import LLMChain
from langchain_community.utilities import SerpAPIWrapper
from langchain_openai import OpenAI

planner_llm = OpenAI(
    temperature=0,
    openai_api_key=OPENAI_API_KEY,
)

todo_prompt = PromptTemplate.from_template(
    """
You are an AI engineering project planner.

Given an objective, create a short, ordered TODO list that focuses on:
1. the user problem and success criteria,
2. relevant current technical options,
3. architecture and implementation steps,
4. evaluation criteria,
5. risks and trade-offs.

Keep the plan practical and concise.

Objective:
{objective}

TODO list:
"""
)

todo_chain = LLMChain(
    llm=planner_llm,
    prompt=todo_prompt,
)

search = SerpAPIWrapper(
    serpapi_api_key=SERPAPI_API_KEY,
)

tools = [
    Tool(
        name="Web Search",
        func=search.run,
        description=(
            "Use this tool when current external information is required, such as "
            "recent AI frameworks, model capabilities, APIs, benchmarks, or best practices."
        ),
    ),
    Tool(
        name="AI Project Planner",
        func=todo_chain.run,
        description=(
            "Use this tool to break an AI engineering objective into a concise implementation, "
            "evaluation, and risk-management plan. Input must be the complete objective."
        ),
    ),
]

prefix = """
You are an AI engineering research agent.

Your overall objective is:
{objective}

Complete the current task using reliable information. Take into account the results of
previously completed tasks:

{context}

Rules:
- Use Web Search whenever the task depends on current external facts.
- Use AI Project Planner when decomposition or implementation planning is needed.
- Distinguish facts from recommendations.
- Prefer a simple architecture over unnecessary complexity.
- Mention important trade-offs when relevant.
"""

suffix = """
Current task:
{task}

{agent_scratchpad}
"""

prompt = ZeroShotAgent.create_prompt(
    tools,
    prefix=prefix,
    suffix=suffix,
    input_variables=["objective", "task", "context", "agent_scratchpad"],
)


/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `LLMChain` was deprecated in LangChain 0.1.17 and will be removed in 0.3.0. Use RunnableSequence, e.g., `prompt | llm` instead.
  warn_deprecated(


In [12]:
llm = OpenAI(
    temperature=0,
    openai_api_key=OPENAI_API_KEY,
)

llm_chain = LLMChain(
    llm=llm,
    prompt=prompt,
)

tool_names = [tool.name for tool in tools]

agent = ZeroShotAgent(
    llm_chain=llm_chain,
    allowed_tools=tool_names,
)

agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent,
    tools=tools,
    verbose=True,
    handle_parsing_errors=True,
)


/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The class `ZeroShotAgent` was deprecated in LangChain 0.1.0 and will be removed in 0.2.0. Use create_react_agent instead.
  warn_deprecated(


### Run the BabyAGI

Now it's time to create the BabyAGI controller and watch it try to accomplish your objective.

In [14]:
OBJECTIVE = """
Create a concise technical recommendation for a small company that wants to build
a customer-support RAG chatbot over its PDF and Word documents.

The recommendation must identify:
- a simple architecture,
- suitable current tools/frameworks,
- how documents should be ingested and retrieved,
- how answer quality should be evaluated,
- the main production risks and trade-offs.

Keep the proposed MVP realistic for a small engineering team.
"""


In [15]:
# Logging of LLMChains
verbose = True

# Keep the autonomous loop bounded for the lab.
max_iterations: Optional[int] = 4

baby_agi = BabyAGI.from_llm(
    llm=llm,
    vectorstore=vectorstore,
    task_execution_chain=agent_executor,
    verbose=verbose,
    max_iterations=max_iterations,
)


In [16]:
# Run the autonomous agent.
# Some older BabyAGI versions are callable, while newer LangChain interfaces prefer invoke().
if hasattr(baby_agi, "invoke"):
    result = baby_agi.invoke({"objective": OBJECTIVE})
else:
    result = baby_agi({"objective": OBJECTIVE})

result


/opt/anaconda3/lib/python3.11/site-packages/langchain_core/_api/deprecation.py:119: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain 0.1.0 and will be removed in 0.2.0. Use invoke instead.
  warn_deprecated(



*****TASK LIST*****

1: Make a todo list

*****NEXT TASK*****

1: Make a todo list


> Entering new AgentExecutor chain...
Thought: I should create a todo list based on the given objective
Action: TODO
Action Input: Create a todo list for SF weather report
Observation: 

1. Research the current weather patterns in San Francisco.
2. Determine the sources for obtaining accurate weather information for the city.
3. Create a list of key weather elements to include in the report, such as temperature, precipitation, wind speed, and humidity.
4. Decide on the format of the report, whether it will be a written document, a presentation, or a combination of both.
5. Set a timeline for when the report needs to be completed and shared.
6. Gather data from reliable sources, such as the National Weather Service or local meteorologists.
7. Organize the data into a clear and easy-to-understand format.
8. Include any relevant graphics or visuals to enhance the report.
9. Proofread and edit the report 

{'objective': 'Write a weather report for SF today'}

## Lab Reflection

I changed the original objective from a simple weather report to an **AI-engineering research and architecture task**: planning a small customer-support RAG chatbot.

The tools were adapted to match that objective:

- **Web Search** provides current information when the agent needs to investigate frameworks, model capabilities, APIs, or industry practices.
- **AI Project Planner** decomposes the objective into implementation, evaluation, and risk-management steps.

The execution prompt was also changed so that the agent behaves like an AI engineering researcher rather than a generic task executor. It is instructed to prefer simple architectures, separate facts from recommendations, use search for current information, and mention trade-offs.

### Why BabyAGI is useful here

BabyAGI automatically creates, prioritizes, and executes tasks while storing previous results in a vector store. This is useful for open-ended research objectives because later tasks can incorporate earlier findings.

### Advantages

- It can decompose an ambiguous objective into smaller tasks.
- The execution agent can use external tools instead of relying only on LLM memory.
- The vector store gives the loop a basic form of working memory.
- Bounding the run with `max_iterations` makes the experiment manageable.

### Limitations

- Autonomous loops can create redundant or low-value tasks.
- Every iteration and web search adds latency and API cost.
- Tool selection and task prioritization are probabilistic.
- Search results can still be unreliable, so production systems need source validation.
- BabyAGI is useful educationally, but for production I would usually prefer an explicit workflow/state graph because it is easier to test, observe, and control.

The main lesson is that autonomous-agent quality depends not only on the LLM, but also on the **objective, tool descriptions, planner prompt, memory design, and execution limits**.
